# BioASQ gold PMID → PMC Open Access coverage

Estimate how much of the BioASQ 11b gold-document set can be represented by reusable PMC Open Access full text. The analysis uses **all BioASQ 11b training questions** and reproduces the six question/document-count groups used by `BioASQ_sample.ipynb`:

- `list-one`, `list-multiple`
- `factoid-one`, `factoid-multiple`
- `summary-one`, `summary-multiple`

For every gold PMID the notebook checks two stages:
1. whether the PMID maps to a PMCID (article exists in PMC), and
2. whether that PMCID belongs to the **PMC Open Access Subset**.

The second criterion is the relevant one for building a redistributable/text-mining full-text benchmark. Coverage is reported both per unique gold document and per query-document relevance relation, plus the fraction of questions for which at least one / all gold documents are OA.

Official services used: [PMC ID Converter API](https://pmc.ncbi.nlm.nih.gov/tools/id-converter-api/) and [NCBI E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/).


In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/lohex/retrieval-benchlab.git"
REPO_BRANCH = "main"
REPO_ROOT = Path("/content/retrieval-benchlab")

if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} fetch origin {REPO_BRANCH}
        !git -C {REPO_ROOT} checkout {REPO_BRANCH}
        !git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U pandas requests tqdm


In [ ]:
import json
import time
from collections import Counter

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

from src.io import load_bioasq_questions, mount_google_drive

mount_google_drive()


## Configuration

NCBI asks API clients to identify themselves with a tool name and maintainer email. An API key is optional; without one the notebook stays below the normal E-utilities request rate. Results are cached on Google Drive so repeated runs do not repeat completed mappings.


In [ ]:
QUESTIONS_SOURCE = "https://zenodo.org/api/records/7655130/files/training11b.json/content"
QUESTION_TYPES = ("list", "factoid", "summary")

OUTPUT_DIR = Path("/content/drive/MyDrive/Retreaval/pmc_oa_coverage")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = OUTPUT_DIR / "bioasq11b_pmc_oa_mapping.csv"
SUMMARY_PATH = OUTPUT_DIR / "bioasq11b_pmc_oa_summary.csv"

NCBI_TOOL = "retrieval_benchlab"
NCBI_EMAIL = ""  # optional: set your email to follow NCBI API guidance
NCBI_API_KEY = ""  # optional

IDCONV_BATCH_SIZE = 200
OA_SEARCH_BATCH_SIZE = 100
REQUEST_DELAY_SECONDS = 0.36 if not NCBI_API_KEY else 0.11


## Load BioASQ questions and reproduce the six splits

`one` means exactly one annotated gold document; `multiple` means more than one. Yes/no questions are excluded because the existing sample benchmark uses `list`, `factoid`, and `summary`.


In [ ]:
queries, relevant_docs, query_types = load_bioasq_questions(QUESTIONS_SOURCE)

question_rows = []
for qid, question_type in query_types.items():
    if question_type not in QUESTION_TYPES:
        continue
    gold_pmids = sorted(relevant_docs[qid])
    cardinality = "one" if len(gold_pmids) == 1 else "multiple"
    split = f"{question_type}-{cardinality}"
    question_rows.append({
        "qid": qid,
        "question_type": question_type,
        "cardinality": cardinality,
        "split": split,
        "n_gold_docs": len(gold_pmids),
        "gold_pmids": gold_pmids,
    })

questions_df = pd.DataFrame(question_rows)
split_order = [
    "list-one", "list-multiple",
    "factoid-one", "factoid-multiple",
    "summary-one", "summary-multiple",
]
questions_df["split"] = pd.Categorical(questions_df["split"], categories=split_order, ordered=True)

split_counts = (
    questions_df.groupby("split", observed=False)
    .agg(questions=("qid", "nunique"), gold_relations=("n_gold_docs", "sum"))
    .reset_index()
)
display(split_counts)


## Map unique gold PMIDs to PMCIDs

The PMC ID Converter accepts up to 200 IDs per request. Missing PMCIDs are retained as explicit negative mappings rather than dropped.


In [ ]:
def batched(values, size):
    values = list(values)
    for start in range(0, len(values), size):
        yield values[start:start + size]


def build_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "POST"}),
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


session = build_session()

all_gold_pmids = sorted(set().union(*questions_df["gold_pmids"].map(set)))
print(f"Unique gold PMIDs: {len(all_gold_pmids):,}")

existing_mapping = pd.DataFrame()
if CACHE_PATH.exists():
    existing_mapping = pd.read_csv(CACHE_PATH, dtype={"pmid": str, "pmcid": str})
    print(f"Loaded {len(existing_mapping):,} cached mappings from {CACHE_PATH}")

cached_pmids = set(existing_mapping.get("pmid", pd.Series(dtype=str)).astype(str))
pmids_to_map = [pmid for pmid in all_gold_pmids if pmid not in cached_pmids]

idconv_url = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
new_records = []
for batch in tqdm(list(batched(pmids_to_map, IDCONV_BATCH_SIZE)), desc="PMID → PMCID"):
    params = {
        "ids": ",".join(batch),
        "idtype": "pmid",
        "format": "json",
        "tool": NCBI_TOOL,
    }
    if NCBI_EMAIL:
        params["email"] = NCBI_EMAIL
    response = session.get(idconv_url, params=params, timeout=120)
    response.raise_for_status()
    payload = response.json()
    by_requested_id = {str(record.get("requested-id")): record for record in payload.get("records", [])}
    for pmid in batch:
        record = by_requested_id.get(str(pmid), {})
        new_records.append({
            "pmid": str(pmid),
            "pmcid": record.get("pmcid"),
            "doi": record.get("doi"),
        })
    time.sleep(REQUEST_DELAY_SECONDS)

new_mapping = pd.DataFrame(new_records)
mapping_df = pd.concat([existing_mapping, new_mapping], ignore_index=True)
if mapping_df.empty:
    mapping_df = pd.DataFrame(columns=["pmid", "pmcid", "doi", "pmc_open_access"])
mapping_df["pmid"] = mapping_df["pmid"].astype(str)
mapping_df = mapping_df.drop_duplicates("pmid", keep="last")
mapping_df = mapping_df[mapping_df["pmid"].isin(all_gold_pmids)].copy()
mapping_df["in_pmc"] = mapping_df["pmcid"].notna() & mapping_df["pmcid"].astype(str).ne("nan")
mapping_df.head()


## Identify PMC Open Access articles

PMC ESearch supports the `open access[filter]` filter. We query PMC IDs in small POST batches and mark only IDs returned by that filter as reusable PMC-OA articles. This avoids downloading the full texts just to test availability.


In [ ]:
if "pmc_open_access" not in mapping_df.columns:
    mapping_df["pmc_open_access"] = pd.NA

pmc_rows = mapping_df[mapping_df["in_pmc"]].copy()
needs_oa_check = pmc_rows["pmc_open_access"].isna()
pmcids_to_check = pmc_rows.loc[needs_oa_check, "pmcid"].astype(str).tolist()

esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
oa_pmcids = set(
    mapping_df.loc[mapping_df["pmc_open_access"] == True, "pmcid"].dropna().astype(str)
)

for batch in tqdm(list(batched(pmcids_to_check, OA_SEARCH_BATCH_SIZE)), desc="PMC → OA subset"):
    numeric_ids = [pmcid.removeprefix("PMC") for pmcid in batch]
    id_query = " OR ".join(f"{pmcid}[uid]" for pmcid in numeric_ids)
    params = {
        "db": "pmc",
        "term": f"({id_query}) AND open access[filter]",
        "retmode": "json",
        "retmax": len(batch),
        "tool": NCBI_TOOL,
    }
    if NCBI_EMAIL:
        params["email"] = NCBI_EMAIL
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    response = session.post(esearch_url, data=params, timeout=120)
    response.raise_for_status()
    returned_ids = response.json().get("esearchresult", {}).get("idlist", [])
    oa_pmcids.update(f"PMC{uid}" for uid in returned_ids)
    time.sleep(REQUEST_DELAY_SECONDS)

checked_pmcids = set(pmc_rows["pmcid"].dropna().astype(str))
mapping_df.loc[mapping_df["pmcid"].astype(str).isin(checked_pmcids), "pmc_open_access"] = (
    mapping_df.loc[mapping_df["pmcid"].astype(str).isin(checked_pmcids), "pmcid"]
    .astype(str).isin(oa_pmcids).to_numpy()
)
mapping_df.loc[~mapping_df["in_pmc"], "pmc_open_access"] = False
mapping_df["pmc_open_access"] = mapping_df["pmc_open_access"].fillna(False).astype(bool)
mapping_df.sort_values("pmid").to_csv(CACHE_PATH, index=False)
print(f"Saved mapping cache to {CACHE_PATH}")
print(f"PMC: {mapping_df['in_pmc'].sum():,}/{len(mapping_df):,} ({mapping_df['in_pmc'].mean():.1%})")
print(f"PMC OA: {mapping_df['pmc_open_access'].sum():,}/{len(mapping_df):,} ({mapping_df['pmc_open_access'].mean():.1%})")


## Coverage by BioASQ split

The table distinguishes document-level coverage from query-level usability. `questions_all_gold_oa` is the strictest criterion for reproducing the original relevance set without losing any annotated positive document.


In [ ]:
mapping_by_pmid = mapping_df.set_index("pmid")

detail_rows = []
for row in questions_df.itertuples(index=False):
    for pmid in row.gold_pmids:
        mapped = mapping_by_pmid.loc[str(pmid)]
        detail_rows.append({
            "qid": row.qid,
            "question_type": row.question_type,
            "cardinality": row.cardinality,
            "split": str(row.split),
            "pmid": str(pmid),
            "pmcid": mapped["pmcid"],
            "in_pmc": bool(mapped["in_pmc"]),
            "pmc_open_access": bool(mapped["pmc_open_access"]),
        })
detail_df = pd.DataFrame(detail_rows)
detail_df["split"] = pd.Categorical(detail_df["split"], categories=split_order, ordered=True)

question_coverage = (
    detail_df.groupby(["split", "qid"], observed=False)
    .agg(
        n_gold_docs=("pmid", "size"),
        n_gold_in_pmc=("in_pmc", "sum"),
        n_gold_oa=("pmc_open_access", "sum"),
    )
    .reset_index()
)
question_coverage["any_gold_oa"] = question_coverage["n_gold_oa"] > 0
question_coverage["all_gold_oa"] = question_coverage["n_gold_oa"] == question_coverage["n_gold_docs"]

summary_rows = []
for split in split_order:
    relations = detail_df[detail_df["split"] == split]
    qcov = question_coverage[question_coverage["split"] == split]
    unique_docs = relations.drop_duplicates("pmid")
    n_unique = len(unique_docs)
    n_relations = len(relations)
    n_questions = qcov["qid"].nunique()
    summary_rows.append({
        "split": split,
        "questions": n_questions,
        "unique_gold_pmids": n_unique,
        "gold_relations": n_relations,
        "unique_pmids_in_pmc": int(unique_docs["in_pmc"].sum()),
        "unique_pmids_in_pmc_oa": int(unique_docs["pmc_open_access"].sum()),
        "pmc_coverage_unique": unique_docs["in_pmc"].mean() if n_unique else float("nan"),
        "oa_coverage_unique": unique_docs["pmc_open_access"].mean() if n_unique else float("nan"),
        "oa_coverage_relations": relations["pmc_open_access"].mean() if n_relations else float("nan"),
        "questions_any_gold_oa": int(qcov["any_gold_oa"].sum()),
        "questions_all_gold_oa": int(qcov["all_gold_oa"].sum()),
        "question_any_oa_rate": qcov["any_gold_oa"].mean() if n_questions else float("nan"),
        "question_all_oa_rate": qcov["all_gold_oa"].mean() if n_questions else float("nan"),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
display(summary_df.style.format({
    "pmc_coverage_unique": "{:.1%}",
    "oa_coverage_unique": "{:.1%}",
    "oa_coverage_relations": "{:.1%}",
    "question_any_oa_rate": "{:.1%}",
    "question_all_oa_rate": "{:.1%}",
}).hide(axis="index"))
print(f"Saved summary to {SUMMARY_PATH}")


## Overall benchmark feasibility

This aggregates the six splits. `all_gold_oa` is the conservative estimate of how many BioASQ questions can be transferred to a PMC-OA full-text benchmark **without changing their positive relevance set**.


In [ ]:
overall_unique = detail_df.drop_duplicates("pmid")
overall_questions = question_coverage.drop_duplicates("qid")
overall = pd.Series({
    "questions": overall_questions["qid"].nunique(),
    "unique_gold_pmids": overall_unique["pmid"].nunique(),
    "unique_gold_pmids_in_pmc": int(overall_unique["in_pmc"].sum()),
    "unique_gold_pmids_in_pmc_oa": int(overall_unique["pmc_open_access"].sum()),
    "pmc_coverage_unique": overall_unique["in_pmc"].mean(),
    "oa_coverage_unique": overall_unique["pmc_open_access"].mean(),
    "questions_any_gold_oa": int(overall_questions["any_gold_oa"].sum()),
    "questions_all_gold_oa": int(overall_questions["all_gold_oa"].sum()),
    "question_any_oa_rate": overall_questions["any_gold_oa"].mean(),
    "question_all_oa_rate": overall_questions["all_gold_oa"].mean(),
})
display(overall.to_frame("value"))


## Optional inspection of missing gold documents

Use this table to see which gold PMIDs are absent from PMC entirely versus present in PMC but outside the Open Access Subset.


In [ ]:
missing_df = mapping_df.loc[~mapping_df["pmc_open_access"], ["pmid", "pmcid", "in_pmc", "pmc_open_access", "doi"]].copy()
display(missing_df.head(50))
print(f"Gold PMIDs without PMC-OA full text: {len(missing_df):,}")
